# Lesson10：成果展示与未来展望

本课完成三件事：用可重复的流程展示项目；把 Lesson 09 的单文件程序映射为 ROS 2 Jazzy 节点；理解强化学习如何在仿真中训练运动策略。

本 notebook **不导入宇树 SDK、不初始化机器人客户端、不发送真机指令**。机器人展示使用各组已经验证的项目程序，并按独立的启动与停止清单执行。

## 学习目标

完成本课后，可以：

- 在三分钟内讲清问题、架构、运行结果、安全设计和局限。
- 使用展示前检查、明确的停止条件和备用证据降低现场失败风险。
- 把摄像头、LiDAR、决策、动作和日志映射为 ROS 2 节点与通信机制。
- 解释强化学习中的状态、动作、奖励和策略。
- 区分仿真训练、策略评估和受限真机部署，避免真机探索训练。

## 第三天下午节奏

- 45 分钟：项目整合、异常测试和展示彩排。
- 20 分钟：展示结构、评价标准和故障备用方案。
- 60 分钟：分组展示、同伴反馈与集中答疑。
- 20 分钟：ROS 2 Jazzy 架构映射。
- 15 分钟：强化学习与 sim-to-real 路线。
- 10 分钟：复盘和后续学习路径。

## 10.0 三分钟讲清一个机器人项目

1. **问题**：机器人服务谁，完成什么任务。
2. **系统**：输入、判断、输出和安全层如何连接。
3. **运行**：用一次完整闭环证明系统可工作。
4. **安全与隐私**：如何停车、降级，以及是否保存现场数据。
5. **反思**：当前失败边界和最值得继续改进的一项能力。

展示不是动作合集。每组只保留一条清晰主线，并为现场故障准备录屏、日志或视频。

## 10.1 展示清单与项目清单

展示前必须明确机器人平台、入口程序、正常停止方法、紧急停止方法、数据策略和备用证据。下面的代码只检查清单完整性。

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable


@dataclass(frozen=True)
class ProjectManifest:
    team: str
    project: str
    robot_type: str
    entry_command: str
    normal_stop: str
    emergency_stop: str
    data_policy: str
    fallback_evidence: str


def validate_manifest(manifest: ProjectManifest) -> list[str]:
    errors: list[str] = []
    if manifest.robot_type not in {"go2", "g1"}:
        errors.append("robot_type 必须是 go2 或 g1")
    for field_name, value in manifest.__dict__.items():
        if not str(value).strip():
            errors.append(f"{field_name} 不能为空")
    if "StopMove" not in manifest.normal_stop:
        errors.append("正常停止说明应明确包含 StopMove")
    return errors


MANIFEST = ProjectManifest(
    team="示例组",
    project="安全迎宾伙伴",
    robot_type="go2",
    entry_command="python project.py --dry-run",
    normal_stop="程序 finally 调用 StopMove()",
    emergency_stop="遥控器进入现场规定的安全状态",
    data_policy="只识别通用 person 类别，不保存原始人脸",
    fallback_evidence="已验证的录屏和日志",
)

errors = validate_manifest(MANIFEST)
print("项目清单：", "通过" if not errors else errors)

## 10.2 展示前检查与运行表

机器人一次只运行一台。检查电量、网络、地面、机械状态、隔离线和遥控器后，先验证停车，再运行主流程。下一段代码模拟展示运行表，不调用机器人。

In [ ]:
@dataclass(frozen=True)
class RunbookStep:
    order: int
    name: str
    pass_condition: str
    failure_action: str


RUNBOOK = [
    RunbookStep(1, "清空并隔离场地", "机器人活动区无人且无松散物", "不开始展示"),
    RunbookStep(2, "检查电量与网络", "状态正常且有线接口正确", "切换备用设备或视频"),
    RunbookStep(3, "验证停止", "StopMove 与遥控安全状态有效", "立即结束真机展示"),
    RunbookStep(4, "运行最小闭环", "单次任务完成且日志完整", "停车并展示日志"),
    RunbookStep(5, "结束与复位", "机器人停车、动作释放、数据按策略处理", "人工执行安全关停"),
]

print("离线展示运行表：")
for step in RUNBOOK:
    print(f"{step.order}. {step.name} | 通过：{step.pass_condition} | 失败：{step.failure_action}")

## 10.3 成果评价

| 维度 | 比例 | 可观察证据 |
|---|---:|---|
| 任务完成度 | 25% | 场景目标完成，流程有开始和结束 |
| 安全与故障处理 | 25% | 停止、超时、失联、异常和隐私策略 |
| 系统整合 | 20% | 感知、决策与执行边界清楚 |
| 稳定复现 | 20% | 多次运行结果一致，有日志和备用证据 |
| 表达与反思 | 10% | 能说明设计取舍、局限与改进方向 |

## 10.4 用 ROS 2 Jazzy 拆分项目

ROS 2 是机器人软件中间件与工具集合，不是替代 Ubuntu 的传统操作系统。课程镜像采用 Ubuntu 24.04 时，可使用与平台适配并提前验证的 ROS 2 Jazzy。Jazzy 是长期支持版本；不要继续把已经结束支持的 Iron 作为新课程默认选项。

- **Topic**：持续数据流，例如图像、点云、检测结果和机器人状态。
- **Service**：一次请求与响应，例如查询模式或保存配置。
- **Action**：有反馈、可取消的长期任务，例如导航到目标点。
- **Node**：把感知、决策、控制和日志拆成独立进程。

In [ ]:
ROS_NODES = {
    "camera_node": "发布 /camera/image_raw",
    "lidar_node": "发布 /scan 或距离摘要",
    "detector_node": "订阅图像，发布 /detections",
    "safety_node": "订阅距离，发布唯一安全运动命令",
    "task_node": "订阅检测与安全状态，管理任务状态机",
    "robot_bridge": "把受限命令转换为厂商 SDK 调用",
    "logger_node": "记录状态、错误和关键传感器数据",
}

ROS_EDGES = [
    ("camera_node", "detector_node", "Topic: image"),
    ("lidar_node", "safety_node", "Topic: ranges"),
    ("detector_node", "task_node", "Topic: detections"),
    ("safety_node", "task_node", "Topic: safety_state"),
    ("task_node", "safety_node", "Topic: desired_motion"),
    ("safety_node", "robot_bridge", "Topic: safe_motion"),
]

print("ROS 2 Jazzy 项目映射：")
for source, target, channel in ROS_EDGES:
    print(f"{source:14s} -- {channel:21s} --> {target}")

## 10.5 ROS 2 十分钟观察演示

在课程已准备好的 Jazzy 环境中依次观察：

```bash
source /opt/ros/jazzy/setup.bash
ros2 node list
ros2 topic list
ros2 topic echo /project/state
rqt_graph
ros2 bag record /project/state /detections
```

这些命令只在已经核对话题名称和数据策略的环境中运行。涉及图像与人脸的数据默认不写入 bag。

## 10.6 强化学习：从规则到策略

- **智能体**：机器人控制策略。
- **环境**：仿真器中的机器人、地面和扰动。
- **状态**：关节、机身姿态、速度、触地信息和任务目标。
- **动作**：仿真中的目标关节位置、速度或受约束控制量。
- **奖励**：速度跟踪、姿态稳定、能耗、碰撞和跌倒等目标的组合。
- **策略**：从状态映射到动作的函数。

本课只分析离线轨迹的奖励，不训练模型，也不向 Go2 或 G1 输出任何动作。

In [ ]:
@dataclass(frozen=True)
class SimStep:
    target_speed: float
    actual_speed: float
    tilt_rad: float
    energy: float
    collision: bool = False
    fallen: bool = False


def locomotion_reward(step: SimStep) -> float:
    """离线奖励示例，不连接仿真器或机器人。"""
    tracking = math.exp(-4.0 * (step.actual_speed - step.target_speed) ** 2)
    posture_penalty = 0.8 * abs(step.tilt_rad)
    energy_penalty = 0.02 * max(step.energy, 0.0)
    collision_penalty = 3.0 if step.collision else 0.0
    fall_penalty = 10.0 if step.fallen else 0.0
    return tracking - posture_penalty - energy_penalty - collision_penalty - fall_penalty


SAFE_TRACE = [
    SimStep(0.5, 0.46, 0.03, 4.0),
    SimStep(0.5, 0.51, 0.04, 4.3),
    SimStep(0.5, 0.49, 0.03, 4.1),
]
UNSAFE_TRACE = [
    SimStep(0.5, 0.62, 0.25, 8.0),
    SimStep(0.5, 0.20, 0.55, 10.0, collision=True),
    SimStep(0.5, 0.00, 1.10, 3.0, fallen=True),
]

for name, trace in (("稳定轨迹", SAFE_TRACE), ("危险轨迹", UNSAFE_TRACE)):
    total = sum(locomotion_reward(step) for step in trace)
    print(f"{name}累计奖励：{total:.3f}")

## 10.7 从仿真到真机的受控流程

```text
任务定义 → 仿真建模 → 奖励与约束 → 大规模训练
        → 随机化与扰动测试 → 离线评估 → 安全限幅
        → 吊装/隔离条件下小范围部署 → 逐级扩大验证
```

域随机化可以改变质量、摩擦、延迟和传感器噪声，降低仿真与真实世界差异。但它不能提供形式化安全保证。部署仍需要速度与关节限幅、异常监测、厂商底层保护、机械隔离和人工急停。

禁止在培训现场使用真机进行探索式强化学习，禁止直接输出未经验证的关节扭矩策略。

## 10.8 从课程项目继续前进

1. **反馈闭环**：为现有项目补充超时、重试、状态确认和可观测日志。
2. **ROS 2 模块化**：把传感器、任务、安全和机器人桥接拆成节点。
3. **导航与操作**：学习 SLAM、Nav2、MoveIt 2 或人形遥操作。
4. **仿真与学习**：在仿真器中建立奖励、随机化和评估管线。
5. **系统工程**：固定依赖、自动测试、记录数据版本和部署清单。

每组为自己的项目画一张 ROS 2 节点图，并写出三项奖励、三项惩罚和三项禁止越过的安全约束。

## 课程总结

前三课建立连接和基本运动，Lesson 04–07 加入动作、状态、视觉和 LiDAR，Lesson 08 用相似的高层控制方式让 G1 移动并加入双臂动作，Lesson 09 完成安全闭环或移动—操作项目，Lesson 10 再把项目抽象为 ROS 2 系统与仿真学习路线。

最终能力不是“让机器人做一个动作”，而是让机器人任务具备明确输入、可解释决策、受限输出、可靠停止和可重复验证。